# LLMON core use cases

LLMON splits a prompt into explicit channels (`instr` for what to do, `data` for what to
operate on, `req` for output rules) using special tokens, so the model knows what to execute
versus what to only read.

Each case below passes a list of messages to the model's own chat template. A user message
carries `instruction`, `data` and `requirement` lists, and `"exec": True` marks a block as one
the model should act on. Retrieved documents and tool definitions are passed to the template
as `documents` and `tools`.

Requires access to
[`ibm-granite/granite-4.1-3b-llmon-preview`](https://huggingface.co/ibm-granite/granite-4.1-3b-llmon-preview)
(`hf auth login`) and `pip install -r ../requirements.txt`. Weights (~6 GB) download on first run.

In [1]:
import os, torch, transformers

MODEL_ID = os.environ.get("LLMON_MODEL", "ibm-granite/granite-4.1-3b-llmon-preview")
REVISION = "273198fbfd138f2a33506bb47fd7dfc4b2189d7a"  # bf16 weights (7.3 GB); the latest revision stores float32 (14.6 GB)

tok = transformers.AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.bfloat16, device_map="auto").eval()


def run(messages, documents=None, tools=None, max_new_tokens=512):
    """Render the messages with the model's chat template, then generate the reply."""
    prompt = tok.apply_chat_template(messages, documents=documents, tools=tools,
                                     tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()


print("model:", MODEL_ID, "| device:", model.device)

model: ibm-granite/granite-4.1-3b-llmon-preview | device: mps:0


## 1. Extraction
Pull structured values out of free text.

In [2]:
messages = [{
    "role": "user",
    "instruction": [{"value": "Extract the customer's email and phone number from the text below.", "exec": True}],
    "data": [{"value": "Hi, I'm Dana Reyes. Reach me at dana.reyes@example.com or 555-204-8890 after noon.", "exec": True}],
}]
print(run(messages))

Email: dana.reyes@example.com
Phone Number: 555-204-8890


## 2. Classification
Assign a label to supplied content.

In [3]:
messages = [{
    "role": "user",
    "instruction": [{"value": "Classify the sentiment as positive, negative, or mixed.", "exec": True}],
    "data": [{"value": "The screen is gorgeous but it overheats within minutes of gaming.", "exec": True}],
}]
print(run(messages))

Mixed


## 3. RAG (grounded Q&A)
The retrieved context is passed as `documents`, which the template renders as `doc` blocks.

In [4]:
documents = [{
    "title": "Refund policy",
    "text": ("Refunds are available within 30 days of purchase. Items must be unused and in "
             "original packaging. Shipping fees are non-refundable."),
}]
messages = [{
    "role": "user",
    "instruction": [{"value": "Answer the question using ONLY the documents provided. "
                              "Question: How long do I have to request a refund?", "exec": True}],
    "requirement": [{"value": "If the answer is not in the documents, say so.", "exec": True}],
}]
print(run(messages, documents=documents))

According to the refund policy, you have 30 days from the purchase date to request a refund.


## 4. Translation
Transform supplied text.

In [5]:
messages = [{
    "role": "user",
    "instruction": [{"value": "Translate the text to French.", "exec": True}],
    "data": [{"value": "The quick brown fox jumps over the lazy dog.", "exec": True}],
}]
print(run(messages))

Le renard brun rapide saute par-dessus le chien paresseux.


## 5. Rewrite (tone)
The style rule is a `req`, not part of the instruction.

In [6]:
messages = [{
    "role": "user",
    "instruction": [{"value": "Rewrite the message below.", "exec": True}],
    "data": [{"value": "No. We can't do that. Read the manual.", "exec": True}],
    "requirement": [{"value": "Make it polite and professional.", "exec": True}],
}]
print(run(messages))

Thank you for your request. Unfortunately, we are unable to accommodate that particular request. Please refer to the user manual for further guidance.


## 6. Tool calling
The tool is passed as `tools`, in the standard OpenAI format, and the template flattens it into `tool` blocks.

In [7]:
tools = [{
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluates an arithmetic expression",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "The expression to evaluate"}},
            "required": ["expression"],
        },
    },
}]
messages = [{"role": "user", "content": "What is 847 multiplied by 293?"}]
print(run(messages, tools=tools))

<tool_call>{"name": "calculator", "arguments": {"expression": "847 * 293"}}</tool_call>


## 7. Multi-turn
Prior turns are replayed as real messages, and each assistant turn is closed with the end-of-text token.

In [8]:
messages = [
    {"role": "user",
     "instruction": [{"value": "What is your refund policy?", "exec": True}]},
    {"role": "assistant",
     "content": "Refunds are accepted within 30 days of purchase with original receipt. Items must be unused."},
    {"role": "user",
     "instruction": [{"value": "How many days do I have?", "exec": True}]},
]
print(run(messages))

You have 30 days from the date of purchase to request a refund.
